In [8]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
from scipy.signal import find_peaks
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
import matplotlib.patches as patches

# Definizione delle regioni e dei canali
channel_to_region = {
    'E36': 'Frontal',
    'E224': 'Frontal',
    'E59': 'Central',
    'E183': 'Central',
    'E116': 'Occipital'
}

# Mantieni funzioni di base per caricare e processare i dati
def read_so_power_data(file_path):
    try:
        # Read the CSV file
        data = pd.read_csv(file_path)
        
        # Check if the required columns are present
        if "Time (s)" in data.columns and "SO-Power" in data.columns:
            return data
        else:
            print(f"Required columns not found in {file_path}")
            return None
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def read_sleep_stages(file_path, sleep_stage_map):
    try:
        # Read the NPY file
        stages = np.load(file_path)
        
        # Convert to DataFrame with time index
        epoch_duration = 30  # seconds
        time_points = np.arange(len(stages)) * epoch_duration
        
        # Create DataFrame
        stages_df = pd.DataFrame({
            "Time (s)": time_points,
            "Sleep Stage": stages,
            "Stage Name": [sleep_stage_map.get(int(s), "Unknown") for s in stages]
        })
        
        return stages_df
    except Exception as e:
        print(f"Error reading sleep stages from {file_path}: {e}")
        return None

def load_all_subjects_multi_channel(base_path, groups, sleep_stage_map, channel_list):
    """
    Funzione modificata per caricare dati da più canali per ogni soggetto
    """
    all_data = {group: {} for group in groups}
    
    for group in groups:
        group_path = os.path.join(base_path, group)
        
        if not os.path.exists(group_path):
            print(f"Group path {group_path} does not exist")
            continue
        
        # List all subjects in this group
        subject_dirs = [d for d in os.listdir(group_path) 
                      if os.path.isdir(os.path.join(group_path, d))]
        
        for subject in subject_dirs:
            subject_path = os.path.join(group_path, subject)
            
            # Trova il file degli stadi in modo più flessibile
            stages_files = [f for f in os.listdir(subject_path) if f.endswith('_stages.npy')]
            
            if not stages_files:
                print(f"Sleep stages file missing for {group} - {subject}")
                continue
            
            stages_file = os.path.join(subject_path, stages_files[0])
            
            # Carica i dati degli stadi
            stages_data = read_sleep_stages(stages_file, sleep_stage_map)
            if stages_data is None:
                print(f"Error loading sleep stages for {group} - {subject}")
                continue
            
            # Inizializza il dizionario per questo soggetto
            all_data[group][subject] = {
                "sleep_stages": stages_data,
                "channels": {}
            }
            
            # Carica i dati per ciascun canale
            for channel in channel_list:
                # Cerca file con pattern più flessibile
                so_file_patterns = [
                    f"SO_power_none_f_{channel}.csv",
                    f"SO_power_none_c_{channel}.csv",
                    f"SO_power_none_o_{channel}.csv",
                    f"*SO_power*{channel}*.csv"  # Pattern più generale
                ]
                
                found_file = False
                for pattern in so_file_patterns:
                    matching_files = glob.glob(os.path.join(subject_path, pattern))
                    if matching_files:
                        so_file = matching_files[0]
                        so_data = read_so_power_data(so_file)
                        if so_data is not None:
                            all_data[group][subject]["channels"][channel] = so_data
                            print(f"Loaded data for {group} - {subject} - {channel} from {os.path.basename(so_file)}")
                            found_file = True
                            break
                
                if not found_file:
                    print(f"SO power file missing for {group} - {subject} - {channel}")
            
            # Se non ci sono dati di canale, rimuovi questo soggetto
            if not all_data[group][subject]["channels"]:
                del all_data[group][subject]
                print(f"No channel data available for {group} - {subject}, removed from dataset")
    
    return all_data

def normalize_time_multi_channel(subject_data, normalize_nrem_only=False, nrem_stages=[1, 2, 3]):
    """
    Funzione modificata per normalizzare dati di più canali
    """
    for group in subject_data:
        for subject, data in subject_data[group].items():
            # Get sleep stages data
            stages_data = data["sleep_stages"].copy()
            
            # Normalize each channel
            for channel, so_data in data["channels"].items():
                so_data = so_data.copy()
                
                # Filter to times in common between SO power and sleep stages
                min_time = so_data["Time (s)"].min()
                max_time = so_data["Time (s)"].max()
                
                stages_filtered = stages_data[
                    (stages_data["Time (s)"] >= min_time) & 
                    (stages_data["Time (s)"] <= max_time)
                ].copy()
                
                # Get times for normalization (NREM only or entire night)
                if normalize_nrem_only:
                    # Filter to only NREM stages
                    normalization_epochs = stages_filtered[stages_filtered["Sleep Stage"].isin(nrem_stages)]
                    print(f"{group} - {subject} - {channel}: Using {len(normalization_epochs)} NREM epochs for time normalization")
                    
                    # Skip if no NREM data
                    if len(normalization_epochs) == 0:
                        print(f"No NREM data found for {group} - {subject} - {channel}, skipping normalization")
                        continue
                    
                    # Get NREM times for normalization
                    normalization_times = normalization_epochs["Time (s)"].values
                    # Sort the times
                    normalization_times = np.sort(normalization_times)
                    
                    # Create a mapping from time to sequence (in NREM)
                    sequence_map = {t: i for i, t in enumerate(normalization_times)}
                    
                    # Map SO power times to NREM sequence
                    so_data["NREM_Sequence"] = so_data["Time (s)"].map(lambda t: sequence_map.get(t, np.nan))
                    
                    # Drop rows with NaN sequence (not in NREM)
                    so_data = so_data.dropna(subset=["NREM_Sequence"])
                    
                    # Normalize the sequence to 0-100%
                    so_data["Normalized Time"] = (so_data["NREM_Sequence"] / (len(normalization_times) - 1)) * 100
                    
                    # Also normalize stages data for this channel
                    stages_filtered["NREM_Sequence"] = stages_filtered["Time (s)"].map(lambda t: sequence_map.get(t, np.nan))
                    stages_filtered = stages_filtered.dropna(subset=["NREM_Sequence"])
                    stages_filtered["Normalized Time"] = (stages_filtered["NREM_Sequence"] / (len(normalization_times) - 1)) * 100
                    
                    normalization_label = "NREM Time"
                else:
                    # Full night normalization
                    min_time = so_data["Time (s)"].min()
                    max_time = so_data["Time (s)"].max()
                    total_duration = max_time - min_time
                    
                    # Create normalized time (0-100%) for SO power
                    so_data["Normalized Time"] = ((so_data["Time (s)"] - min_time) / total_duration) * 100
                    
                    # Create normalized time for sleep stages
                    stages_filtered["Normalized Time"] = ((stages_filtered["Time (s)"] - min_time) / total_duration) * 100
                    
                    normalization_label = "Full Night"
                
                # Update the channel data
                subject_data[group][subject]["channels"][channel] = so_data
                
                # Store normalized stages data for this channel
                subject_data[group][subject][f"stages_{channel}"] = stages_filtered
    
    return subject_data, normalization_label

def calculate_deep_sleep_segment_power_multi_channel(subject_data, num_segments=10, include_n2=False):
    """
    Funzione modificata per calcolare la potenza del segmento per più canali
    """
    # Initialize data structure for all channels
    deep_sleep_segment_data = {group: {} for group in subject_data}
    
    # Define which sleep stages to include as "Deep Sleep"
    if include_n2:
        deep_sleep_values = [2, 3]  # Include both N2 and N3
        stage_name = "N2+N3"
    else:
        deep_sleep_values = [3]     # Only N3
        stage_name = "N3"
    
    print(f"Analyzing SO power in {stage_name} across {num_segments} segments for all channels")
    
    for group in subject_data:
        for subject, data in subject_data[group].items():
            deep_sleep_segment_data[group][subject] = {}
            
            # Get sleep stages data
            for channel in data["channels"]:
                # Get SO power data for this channel
                so_data = data["channels"][channel].copy()
                
                # Get corresponding normalized stages data
                if f"stages_{channel}" in data:
                    stages_data = data[f"stages_{channel}"].copy()
                else:
                    print(f"No normalized sleep stages found for {group} - {subject} - {channel}, skipping")
                    continue
                
                # Create segment labels for both datasets
                segment_size = 100 / num_segments
                so_data["Segment"] = (so_data["Normalized Time"] // segment_size).astype(int)
                stages_data["Segment"] = (stages_data["Normalized Time"] // segment_size).astype(int)
                
                # Handle edge case for exactly 100%
                so_data.loc[so_data["Segment"] == num_segments, "Segment"] = num_segments - 1
                stages_data.loc[stages_data["Segment"] == num_segments, "Segment"] = num_segments - 1
                
                # Find Deep Sleep epochs based on clinical scoring
                deep_sleep_epochs = stages_data[stages_data["Sleep Stage"].isin(deep_sleep_values)]
                
                # Create a lookup dictionary for segment -> Deep Sleep status
                deep_sleep_time_dict = {}
                for _, row in deep_sleep_epochs.iterrows():
                    deep_sleep_time_dict[row["Time (s)"]] = True
                
                # Mark SO power points that occur during Deep Sleep
                so_data["Is_Deep_Sleep"] = so_data["Time (s)"].apply(lambda t: deep_sleep_time_dict.get(t, False))
                
                # Calculate average Deep Sleep power for each segment
                deep_sleep_data = so_data[so_data["Is_Deep_Sleep"] == True]
                
                if len(deep_sleep_data) > 0:  # Skip if no Deep Sleep data
                    segment_means = deep_sleep_data.groupby("Segment")["SO-Power"].mean()
                    segment_counts = deep_sleep_data.groupby("Segment")["SO-Power"].count()
                    
                    # Only keep segments with enough data points (e.g., at least 10)
                    valid_segments = segment_counts[segment_counts >= 10].index
                    segment_means = segment_means[segment_means.index.isin(valid_segments)]
                    
                    # Store the results for this channel
                    deep_sleep_segment_data[group][subject][channel] = segment_means
                else:
                    print(f"No {stage_name} data found for {group} - {subject} - {channel}")
    
    return deep_sleep_segment_data, stage_name

def create_regional_summaries(segment_data, channel_to_region):
    """
    Aggregates channel data by region, with averaging when a region has multiple channels
    """
    # Create a mapping from region to channels
    region_to_channels = {}
    for channel, region in channel_to_region.items():
        if region not in region_to_channels:
            region_to_channels[region] = []
        region_to_channels[region].append(channel)
    
    # Initialize regional data structure
    regional_data = {}
    
    for group in segment_data:
        regional_data[group] = {}
        
        for subject in segment_data[group]:
            # Initialize subject data
            if subject not in regional_data[group]:
                regional_data[group][subject] = {}
            
            # Process each region
            for region, channels in region_to_channels.items():
                # Get available channels for this subject in this region
                available_channels = [ch for ch in channels if ch in segment_data[group][subject]]
                
                if not available_channels:
                    continue  # Skip if no data for this region
                
                # Combine data from all available channels for this region
                channel_dfs = []
                for channel in available_channels:
                    # Convert series to DataFrame with channel name as column
                    channel_df = segment_data[group][subject][channel].to_frame()
                    channel_df.columns = [channel]
                    channel_dfs.append(channel_df)
                
                if not channel_dfs:
                    continue
                
                # Concatenate all channel data side by side
                combined_df = pd.concat(channel_dfs, axis=1)
                
                # Calculate mean across channels for this region
                regional_data[group][subject][region] = combined_df.mean(axis=1)
    
    # Create a group summary for each region
    group_regional_summaries = {}
    
    for region in region_to_channels.keys():
        group_regional_summaries[region] = {}
        
        for group in regional_data:
            # Combine all subjects in this group for this region
            subject_data = {}
            for subject in regional_data[group]:
                if region in regional_data[group][subject]:
                    subject_data[subject] = regional_data[group][subject][region]
            
            if not subject_data:
                continue  # Skip if no data for this group-region combination
            
            # Create a DataFrame with subjects as rows
            region_df = pd.DataFrame(subject_data).T
            
            # Calculate group statistics
            group_regional_summaries[region][group] = {
                "mean": region_df.mean(),
                "se": region_df.std() / np.sqrt(region_df.count()),
                "n": len(region_df),
                "count": region_df.count()
            }
    
    return group_regional_summaries, regional_data

def analyze_temporal_patterns_by_region(group_regional_summaries, stage_name="N2+N3", normalization_label="NREM Time", exclude_s10=False, output_dir="./pattern_analysis_regional"):
    """
    Analizza i pattern temporali della potenza delle SO per ciascuna regione cerebrale
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Colori per i gruppi
    colors = {
        "CTL": "green",
        "DNV": "blue",
        "ADV": "orange",
        "DYS": "red"
    }
    
    # Preparazione iniziale della lista per i risultati
    results_data = []
    
    # Analizza ogni regione separatamente
    for region in group_regional_summaries:
        print(f"\nAnalyzing temporal patterns for {region} region...")
        
        # Figura per confronto di tutti i gruppi per questa regione
        fig, ax = plt.subplots(figsize=(16, 9))  # Figura più larga
        
        # Trova i valori min e max per questa regione
        all_values = []
        
        for group in ["CTL", "DNV", "ADV", "DYS"]:
            if group not in group_regional_summaries[region]:
                continue
                
            time_series = group_regional_summaries[region][group]["mean"].values
            all_values.extend(time_series)
        
        if all_values:
            min_y = min(all_values) - 0.5
            max_y = max(all_values) + 3.0
        else:
            min_y, max_y = 0, 25
        
        ax.set_ylim(min_y, max_y)
        
        # Aumenta lo spazio orizzontale per le etichette di declino
        plt.subplots_adjust(right=0.85)  # Riduce la larghezza del plot per far spazio alle etichette
        
        for group in ["CTL", "DNV", "ADV", "DYS"]:
            if group not in group_regional_summaries[region]:
                print(f"Skipping group {group} for region {region} - no data available")
                continue
                
            # Estrai la serie temporale
            time_series = group_regional_summaries[region][group]["mean"].values
            segments = np.array(group_regional_summaries[region][group]["mean"].index)
            
            # Escludi S10 se richiesto
            if exclude_s10 and len(time_series) > 9:
                time_series = time_series[:-1]
                segments = segments[:-1]
            
            # Calcolo dei valori chiave
            initial_value = time_series[0]
            final_value = time_series[-1]
            
            # Analisi del picco
            peak_idx = np.argmax(time_series)
            peak_value = time_series[peak_idx]
            peak_segment = segments[peak_idx] + 1
            
            # Calcolo del tempo relativo
            max_segment = segments.max()
            peak_time_percent = (segments[peak_idx] / max_segment) * 100 if max_segment > 0 else 0
            
            # Analisi della crescita fino al picco
            initial_to_peak_rise_pct = ((peak_value - initial_value) / initial_value) * 100 if initial_value != 0 else 0
            
            # Analisi del declino post-picco
            if peak_idx < len(segments) - 1:
                # Segmenti e valori dalla fase post-picco
                post_peak_segments = segments[peak_idx:]
                post_peak_values = time_series[peak_idx:]
                
                # Calcolo del tasso di declino
                post_peak_reg = LinearRegression().fit(
                    post_peak_segments.reshape(-1, 1),
                    post_peak_values
                )
                post_peak_decline_rate = post_peak_reg.coef_[0]
                
                # Calcolo della percentuale di declino
                post_peak_decline_pct = ((final_value - peak_value) / peak_value) * 100
            else:
                post_peak_decline_rate = 0
                post_peak_decline_pct = 0
            
            # Determinazione del pattern
            if peak_time_percent < 30:  # Picco precoce
                if post_peak_decline_pct < -15:
                    pattern_type = "Early-Peak + Strong Decline"
                elif post_peak_decline_pct < -5:
                    pattern_type = "Early-Peak + Gradual Decline"
                else:
                    pattern_type = "Early-Peak + Maintenance"
            elif peak_time_percent < 70:  # Picco intermedio
                if post_peak_decline_pct < -15:
                    pattern_type = "Mid-Peak + Strong Decline"
                elif post_peak_decline_pct < -5:
                    pattern_type = "Mid-Peak + Gradual Decline"
                else:
                    pattern_type = "Mid-Peak + Maintenance"
            else:  # Picco tardivo
                pattern_type = "Late-Peak"
            
            # Individua oscillazioni
            peak_prominence = 0.05 * np.ptp(time_series)
            peaks, _ = find_peaks(time_series, prominence=peak_prominence)
            if len(peaks) > 1:
                pattern_type += " (Oscillating)"
            
            # Prepara la riga da aggiungere come dizionario
            new_row = {
                "Group": group,
                "Region": region,
                "Initial_Value": initial_value,
                "Peak_Value": peak_value,
                "Peak_Segment": f"S{int(peak_segment)}",
                "Peak_Time_Percent": peak_time_percent,
                "Final_Value": final_value,
                "Time_To_Peak_Pct": peak_time_percent,
                "Initial_To_Peak_Rise_Pct": initial_to_peak_rise_pct,
                "Post_Peak_Decline_Rate": post_peak_decline_rate,
                "Post_Peak_Decline_Pct": post_peak_decline_pct,
                "Pattern_Type": pattern_type
            }
            
            # Aggiungi il nuovo elemento alla lista
            results_data.append(new_row)
            
            # Visualizzazione grafica
            plt.plot(segments + 1, time_series, 'o-',
                    color=colors.get(group, "black"),
                    linewidth=2, markersize=8,
                    label=f"{group}")
            
            # Evidenzia il picco
            plt.plot(peak_segment, peak_value, '*',
                    color=colors.get(group, "black"),
                    markersize=15, markeredgewidth=2)
            
            # Se c'è una fase post-picco, collega con linea tratteggiata
            if peak_idx < len(segments) - 1:
                plt.plot([peak_segment, segments[-1] + 1], [peak_value, time_series[-1]], '--',
                        color=colors.get(group, "black"), linewidth=1.5)
                
                # Posiziona le etichette di declino sul lato destro del grafico, allineate verticalmente
                y_pos = time_series[-1]
                
                # Calcola una posizione x oltre l'ultimo segmento
                x_pos = segments[-1] + 1.3  # Posizione leggermente a destra dell'ultimo segmento
                
                # Crea un annotazione rettangolare sul lato destro
                box_props = {
                    'boxstyle': 'round,pad=0.3',
                    'facecolor': 'white',
                    'edgecolor': colors.get(group, "black"),
                    'alpha': 0.9
                }
                
                plt.annotate(f"{post_peak_decline_pct:.1f}%",
                           xy=(segments[-1] + 1, y_pos),
                           xytext=(x_pos, y_pos),
                           ha='center', va='center',
                           color=colors.get(group, "black"),
                           fontweight='bold',
                           bbox=box_props,
                           arrowprops=dict(arrowstyle='->', color=colors.get(group, "black"), lw=1.5))
        
        # Finalizza il grafico
        plt.xlabel(f"{normalization_label} Segment", fontsize=14)
        plt.ylabel(f"Average SO Power in {stage_name} (dB)", fontsize=14)
        plt.title(f"Temporal Pattern of SO Power - {region} Region", fontsize=16)
        
        # Estendi l'asse x per fare spazio alle etichette
        max_segment = max(segments) + 1
        plt.xlim(0.5, max_segment + 1.5)  # Aggiungi spazio extra a destra
        
        plt.xticks(range(1, len(segments) + 1), [f"S{i+1}" for i in range(len(segments))])
        plt.grid(True, linestyle="--", alpha=0.7)
        
        # Posiziona la legenda in alto a destra, dentro il grafico
        plt.legend(loc='upper right', fontsize=12, framealpha=0.9)
        
        # Salva la figura
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"temporal_pattern_{region.lower()}.png"), dpi=300, bbox_inches="tight")
        plt.close()
    
    # Crea il DataFrame dai dati raccolti
    results = pd.DataFrame(results_data)
    
    # Salva i risultati in un file CSV
    if len(results) > 0:
        results.to_csv(os.path.join(output_dir, f"pattern_results_{stage_name.lower().replace('+', '_')}.csv"), index=False)
        
        # Analisi statistica dei pattern
        pattern_counts = results.groupby(['Group', 'Pattern_Type']).size().unstack(fill_value=0)
        print("\nDistribuzione dei pattern per gruppo:")
        print(pattern_counts)
        
        # Confronto tra gruppi per le metriche chiave
        print("\nStatistiche dei pattern per gruppo:")
        group_stats = results.groupby('Group').agg({
            'Initial_Value': 'mean',
            'Peak_Value': 'mean',
            'Peak_Time_Percent': 'mean',
            'Initial_To_Peak_Rise_Pct': 'mean',
            'Post_Peak_Decline_Pct': 'mean'
        })
        print(group_stats)
        
        # Confronto tra regioni
        print("\nStatistiche dei pattern per regione:")
        region_stats = results.groupby('Region').agg({
            'Initial_Value': 'mean',
            'Peak_Value': 'mean',
            'Peak_Time_Percent': 'mean',
            'Initial_To_Peak_Rise_Pct': 'mean',
            'Post_Peak_Decline_Pct': 'mean'
        })
        print(region_stats)
    
    return results

def analyze_phase_dynamics_by_region(group_regional_summaries, stage_name="N2+N3", normalization_label="NREM Time", exclude_s10=False, output_dir="./phase_analysis_regional"):
    """
    Analizza le dinamiche delle diverse fasi (iniziale, intermedia, finale) per ogni regione cerebrale
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Colori per le fasi
    phase_colors = ['#8dd3c7', '#bebada', '#fb8072']
    
    # Colori per i gruppi
    group_colors = {
        "CTL": "green",
        "DNV": "blue",
        "ADV": "orange",
        "DYS": "red"
    }
    
    # Preparazione iniziale della lista per i risultati
    all_phase_data = []  # Uso una lista invece di un DataFrame vuoto
    
    # Per ogni regione
    for region in group_regional_summaries:
        # Preparazione per i risultati di questa regione
        region_phase_data = []  # Uso una lista invece di un DataFrame vuoto
        
        # Crea una figura con subplot per ogni gruppo
        num_groups = len([g for g in ["CTL", "DNV", "ADV", "DYS"] if g in group_regional_summaries[region]])
        
        if num_groups == 0:
            continue
            
        # Disponi i subplot in modo appropriato
        if num_groups <= 2:
            fig, axes = plt.subplots(1, num_groups, figsize=(num_groups*7, 6), sharex=True, sharey=False)
            if num_groups == 1:
                axes = [axes]  # Garantisce che axes sia sempre una lista
        else:
            fig, axes = plt.subplots(2, 2, figsize=(14, 12), sharex=True, sharey=False)
            axes = axes.flatten()
        
        # Trovare il range globale y per tutti i gruppi in questa regione
        all_min_y = float('inf')
        all_max_y = float('-inf')
        
        # Prima passata: trova il range y globale
        for group in ["CTL", "DNV", "ADV", "DYS"]:
            if group not in group_regional_summaries[region]:
                continue
                
            time_series = group_regional_summaries[region][group]["mean"].values
            # Aggiorna min e max globali
            all_min_y = min(all_min_y, np.min(time_series))
            all_max_y = max(all_max_y, np.max(time_series))
        
        # Aggiungi padding
        y_range = all_max_y - all_min_y
        all_min_y = max(0, all_min_y - 0.1 * y_range)  # Previene valori negativi
        all_max_y = all_max_y + 0.2 * y_range  # Extra spazio per le annotazioni
        
        group_idx = 0
        for group in ["CTL", "DNV", "ADV", "DYS"]:
            if group not in group_regional_summaries[region]:
                continue
                
            # Estrai la serie temporale
            time_series = group_regional_summaries[region][group]["mean"].values
            segments = np.array(group_regional_summaries[region][group]["mean"].index)
            
            # Escludi S10 se richiesto
            if exclude_s10 and len(time_series) > 9:
                time_series = time_series[:-1]
                segments = segments[:-1]
            
            num_segments = len(segments)
            
            # Calcola i punti di transizione tra le fasi
            early_end = num_segments // 3
            mid_end = 2 * (num_segments // 3)
            
            # Indici per ciascuna fase
            early_indices = range(0, early_end)
            mid_indices = range(early_end, mid_end)
            late_indices = range(mid_end, num_segments)
            
            # Estrai i dati per ciascuna fase
            early_segments = segments[early_indices]
            mid_segments = segments[mid_indices]
            late_segments = segments[late_indices]
            
            early_values = time_series[early_indices]
            mid_values = time_series[mid_indices]
            late_values = time_series[late_indices]
            
            # Calcola statistiche per ciascuna fase
            early_mean = np.mean(early_values)
            mid_mean = np.mean(mid_values)
            late_mean = np.mean(late_values)
            
            # Calcola i cambiamenti percentuali e pendenze
            if len(early_values) > 1:
                early_change_pct = ((early_values[-1] - early_values[0]) / early_values[0]) * 100
                early_reg = LinearRegression().fit(early_segments.reshape(-1, 1), early_values)
                early_slope = early_reg.coef_[0]
            else:
                early_change_pct = 0
                early_slope = 0
            
            if len(mid_values) > 1:
                mid_change_pct = ((mid_values[-1] - mid_values[0]) / mid_values[0]) * 100
                mid_reg = LinearRegression().fit(mid_segments.reshape(-1, 1), mid_values)
                mid_slope = mid_reg.coef_[0]
            else:
                mid_change_pct = 0
                mid_slope = 0
            
            if len(late_values) > 1:
                late_change_pct = ((late_values[-1] - late_values[0]) / late_values[0]) * 100
                late_reg = LinearRegression().fit(late_segments.reshape(-1, 1), late_values)
                late_slope = late_reg.coef_[0]
            else:
                late_change_pct = 0
                late_slope = 0
            
            # Prepara la riga da aggiungere come dizionario
            new_row = {
                "Group": group,
                "Region": region,
                "Early_Phase_Segments": f"S{early_segments[0]+1}-S{early_segments[-1]+1}",
                "Early_Phase_Mean": early_mean,
                "Early_Phase_Change_Pct": early_change_pct,
                "Early_Phase_Slope": early_slope,
                "Mid_Phase_Segments": f"S{mid_segments[0]+1}-S{mid_segments[-1]+1}",
                "Mid_Phase_Mean": mid_mean,
                "Mid_Phase_Change_Pct": mid_change_pct,
                "Mid_Phase_Slope": mid_slope,
                "Late_Phase_Segments": f"S{late_segments[0]+1}-S{late_segments[-1]+1}",
                "Late_Phase_Mean": late_mean,
                "Late_Phase_Change_Pct": late_change_pct,
                "Late_Phase_Slope": late_slope
            }
            
            # Aggiungi il nuovo elemento alla lista invece di concatenare DataFrame
            region_phase_data.append(new_row)
            
            # Visualizzazione grafica
            ax = axes[group_idx]
            
            # Imposta limiti y uniformi per tutti i subplot
            ax.set_ylim(all_min_y, all_max_y)
            
            # Plot della serie temporale originale
            ax.plot(segments + 1, time_series, 'o-', color=group_colors.get(group, "black"),
                  linewidth=2, markersize=8, label="Original")
            
            # Evidenzia le tre fasi con colori diversi
            # Fase precoce
            ax.fill_between(early_segments + 1, early_values, alpha=0.3, color=phase_colors[0], label="Early Phase")
            if len(early_segments) > 1:
                # Linea di trend per la fase precoce
                early_x = np.linspace(early_segments[0], early_segments[-1], 100)
                early_y = early_reg.predict(early_x.reshape(-1, 1))
                ax.plot(early_x + 1, early_y, '-', color=phase_colors[0], linewidth=2)
            
            # Fase intermedia
            ax.fill_between(mid_segments + 1, mid_values, alpha=0.3, color=phase_colors[1], label="Mid Phase")
            if len(mid_segments) > 1:
                # Linea di trend per la fase intermedia
                mid_x = np.linspace(mid_segments[0], mid_segments[-1], 100)
                mid_y = mid_reg.predict(mid_x.reshape(-1, 1))
                ax.plot(mid_x + 1, mid_y, '-', color=phase_colors[1], linewidth=2)
            
            # Fase finale
            ax.fill_between(late_segments + 1, late_values, alpha=0.3, color=phase_colors[2], label="Late Phase")
            if len(late_segments) > 1:
                # Linea di trend per la fase finale
                late_x = np.linspace(late_segments[0], late_segments[-1], 100)
                late_y = late_reg.predict(late_x.reshape(-1, 1))
                ax.plot(late_x + 1, late_y, '-', color=phase_colors[2], linewidth=2)
            
            # Etichette e titolo
            ax.set_title(f"{group} - {region}", fontsize=14)
            ax.set_xlabel(f"{normalization_label} Segment", fontsize=12)
            ax.set_ylabel(f"SO Power in {stage_name} (dB)", fontsize=12)
            ax.grid(True, linestyle="--", alpha=0.7)
            
            # Posiziona la legenda fuori dal grafico
            ax.legend(fontsize=10, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4)
            
            # Imposta i tick sull'asse x
            ax.set_xticks(range(1, len(segments) + 1))
            ax.set_xticklabels([f"S{i+1}" for i in range(len(segments))])
            
            # Aggiungi annotazioni per le pendenze più in alto per evitare sovrapposizioni
            if len(early_segments) > 1:
                ax.annotate(f"Slope: {early_slope:.3f}", 
                          xy=(np.mean(early_segments) + 1, np.mean(early_values)),
                          xytext=(0, 30), textcoords="offset points",
                          ha='center', va='bottom',
                          color=phase_colors[0],
                          fontweight='bold')
            
            if len(mid_segments) > 1:
                ax.annotate(f"Slope: {mid_slope:.3f}", 
                          xy=(np.mean(mid_segments) + 1, np.mean(mid_values)),
                          xytext=(0, 30), textcoords="offset points",
                          ha='center', va='bottom',
                          color=phase_colors[1],
                          fontweight='bold')
            
            if len(late_segments) > 1:
                ax.annotate(f"Slope: {late_slope:.3f}", 
                          xy=(np.mean(late_segments) + 1, np.mean(late_values)),
                          xytext=(0, 30), textcoords="offset points",
                          ha='center', va='bottom',
                          color=phase_colors[2],
                          fontweight='bold')
            
            group_idx += 1
        
        # Titolo generale e layout
        plt.suptitle(f"Phase Dynamics of SO Power - {region} Region", fontsize=16)
        plt.tight_layout(rect=[0, 0.1, 1, 0.95])  # Aumentato lo spazio in basso per la legenda
        
        # Salva la figura
        plt.savefig(os.path.join(output_dir, f"phase_dynamics_{region.lower()}.png"), dpi=300, bbox_inches="tight")
        plt.close()
        
        # Aggiungi tutti i dati della regione alla lista principale
        all_phase_data.extend(region_phase_data)
    
    # Crea il DataFrame dai dati raccolti
    all_phase_results = pd.DataFrame(all_phase_data)
    
    # Salvataggio e ulteriori analisi dei risultati
    if len(all_phase_results) > 0:
        # Salva i risultati in un file CSV
        all_phase_results.to_csv(os.path.join(output_dir, f"phase_dynamics_results_{stage_name.lower().replace('+', '_')}.csv"), index=False)
        
        # Analisi comparative aggiuntive
        print("\n===== ANALISI COMPARATIVE DELLE DINAMICHE DI FASE =====")
        
        # Confronto delle pendenze per fase e regione
        for phase in ['Early', 'Mid', 'Late']:
            print(f"\nPendenze della Fase {phase}:")
            slope_by_region_group = all_phase_results.groupby(['Group', 'Region'])[f'{phase}_Phase_Slope'].mean().unstack()
            print(slope_by_region_group)
        
        # Confronto dei cambiamenti percentuali
        for phase in ['Early', 'Mid', 'Late']:
            print(f"\nCambiamenti Percentuali della Fase {phase}:")
            change_by_region_group = all_phase_results.groupby(['Group', 'Region'])[f'{phase}_Phase_Change_Pct'].mean().unstack()
            print(change_by_region_group)
    
    return all_phase_results

def plot_regional_segment_power(group_regional_summaries, stage_name, normalization_label, num_segments=10, output_dir="./plots_regional"):
    """
    Plotta la potenza delle SO per ciascuna regione cerebrale
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Colors for each group
    colors = {
        "CTL": "green",
        "DNV": "blue",
        "ADV": "orange",
        "DYS": "red"
    }
    
    # Plot all regions in separate figures
    for region in group_regional_summaries:
        plt.figure(figsize=(12, 7))
        
        # Plot each group for this region
        for group in ["CTL", "DNV", "ADV", "DYS"]:
            if group not in group_regional_summaries[region]:
                print(f"Skipping group {group} for region {region} - no data available")
                continue
                
            summary = group_regional_summaries[region][group]
            
            # Get x (segment indices) and y (mean power) values
            x = summary["mean"].index
            y = summary["mean"].values
            yerr = summary["se"].values
            
            # Plot with error bars (MODIFICATO: rimosso n=X dalla legenda)
            plt.errorbar(x, y, yerr=yerr, 
                        label=f"{group}", 
                        marker='o', markersize=8, linewidth=2,
                        linestyle='-', color=colors.get(group, "black"))
        
        # Add labels and title
        plt.xlabel(f"{normalization_label} Segment", fontsize=14)
        plt.ylabel(f"Average SO Power in {stage_name} (dB)", fontsize=14)
        plt.title(f"SO Power in {stage_name} - {region} Region", fontsize=16)
        
        # Add x-ticks for segments
        plt.xticks(range(num_segments), [f"S{i+1}" for i in range(num_segments)])
        
        # Add grid and legend
        plt.grid(True, linestyle="--", alpha=0.7)
        plt.legend(fontsize=12)
        
        # Save the figure
        filename = f"so_power_{stage_name.lower().replace('+', '_')}_{region.lower()}_{normalization_label.lower().replace(' ', '_')}.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
        
        plt.close()
    
    # Create a combined plot with all regions for comparison
    regions = list(group_regional_summaries.keys())
    groups = ["CTL", "DNV", "ADV", "DYS"]
    
    # Determine which groups have data
    available_groups = []
    for group in groups:
        for region in regions:
            if group in group_regional_summaries[region]:
                available_groups.append(group)
                break
    
    if available_groups:
        # Plot each group in a separate subplot with all regions
        fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True)
        axes = axes.flatten()
        
        for i, group in enumerate(groups):
            if i >= len(axes) or group not in available_groups:
                continue
                
            ax = axes[i]
            
            # Plot each region for this group
            for region in regions:
                if group not in group_regional_summaries[region]:
                    continue
                    
                summary = group_regional_summaries[region][group]
                
                x = summary["mean"].index
                y = summary["mean"].values
                
                # Use different line styles and markers for different regions (MODIFICATO: rimosso n=X dalla legenda)
                ax.plot(x, y, marker='o', markersize=6, linewidth=2,
                      label=f"{region}")
            
            ax.set_title(f"{group}", fontsize=14)
            ax.set_xlabel(f"{normalization_label} Segment", fontsize=12)
            ax.set_ylabel(f"Average SO Power in {stage_name} (dB)", fontsize=12)
            ax.grid(True, linestyle="--", alpha=0.7)
            ax.legend(fontsize=10)
            
            # Add x-ticks for segments
            ax.set_xticks(range(num_segments))
            ax.set_xticklabels([f"S{i+1}" for i in range(num_segments)])
        
        plt.suptitle(f"SO Power in {stage_name} by Region and Group", fontsize=16)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        
        # Save the combined figure
        filename = f"so_power_{stage_name.lower().replace('+', '_')}_all_regions_by_group.png"
        plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
        
        plt.close()
    
    return

def main_multi_regional(base_path=None, include_n2=True, normalize_nrem_only=True, exclude_s10=False):
    """
    Funzione principale per eseguire l'analisi multi-regionale completa
    """
    # Usa il percorso base predefinito se non specificato
    if base_path is None:
        base_path = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/clinical_scorings"
    
    # Definisci i gruppi
    groups = ["CTL", "DNV", "ADV", "DYS"]
    
    # Definisci la mappatura degli stadi del sonno
    sleep_stage_map = {
        0: "Wake",
        1: "N1",
        2: "N2",
        3: "N3", 
        5: "REM",
        6: "Artifacts"
    }
    
    # Definisci gli stadi NREM per la normalizzazione
    nrem_stages = [1, 2, 3]  # N1, N2, N3
    
    # Elenco dei canali da analizzare
    channel_list = list(channel_to_region.keys())
    
    # Descrizione dei parametri
    stage_desc = "N2+N3" if include_n2 else "N3 only"
    norm_desc = "NREM time" if normalize_nrem_only else "full night"
    
    print(f"\n===== ANALYZING {stage_desc} WITH {norm_desc.upper()} MULTI-REGIONAL ANALYSIS =====\n")
    
    # Debug: Mostra tutti i file per ogni soggetto
    for group in groups:
        group_path = os.path.join(base_path, group)
        for subject in os.listdir(group_path):
            subject_path = os.path.join(group_path, subject)
            if os.path.isdir(subject_path):
                print(f"\nFile in {group} - {subject}:")
                print(os.listdir(subject_path))
    
    # Step 1: Carica tutti i dati dei soggetti per tutti i canali
    print("Caricamento dati dei soggetti e stadi clinici del sonno per tutti i canali...")
    subject_data = load_all_subjects_multi_channel(base_path, groups, sleep_stage_map, channel_list)
    
    # Stampa il riepilogo dei dati caricati
    for group in subject_data:
        print(f"Gruppo {group}: {len(subject_data[group])} soggetti caricati con dati completi")
        for subject in subject_data[group]:
            print(f"  - {subject}: {len(subject_data[group][subject]['channels'])} canali disponibili")
    
    # Step 2: Normalizza il tempo per tutti i canali
    print(f"\nNormalizzazione delle scale temporali utilizzando {norm_desc} per tutti i canali...")
    normalized_data, normalization_label = normalize_time_multi_channel(subject_data, normalize_nrem_only=normalize_nrem_only, nrem_stages=nrem_stages)
    
    # Step 3: Calcola la potenza media del sonno profondo in ciascun segmento per ogni canale
    num_segments = 10
    print(f"\nCalcolo delle medie di potenza {stage_desc} per {num_segments} segmenti in tutti i canali...")
    segment_data, stage_name = calculate_deep_sleep_segment_power_multi_channel(normalized_data, num_segments=num_segments, include_n2=include_n2)
    
    # Step 4: Crea riepiloghi regionali
    print(f"\nCreazione dei riepiloghi regionali per la potenza SO {stage_desc}...")
    group_regional_summaries, regional_data = create_regional_summaries(segment_data, channel_to_region)
    
    # Step 5: Visualizza i dati regionali
    print(f"\nRappresentazione grafica dei pattern di potenza SO {stage_desc} per regione...")
    output_base_dir = f"./results_regional_{stage_name.lower().replace('+', '_')}_{normalization_label.lower().replace(' ', '_')}"
    os.makedirs(output_base_dir, exist_ok=True)
    plot_regional_segment_power(group_regional_summaries, stage_name, normalization_label, num_segments=num_segments, output_dir=output_base_dir)
    
    # Step 6: Analizza i pattern temporali per regione
    print(f"\n===== ANALISI DEI PATTERN REGIONALI =====")
    pattern_dir = os.path.join(output_base_dir, "pattern_analysis")
    pattern_results = analyze_temporal_patterns_by_region(
        group_regional_summaries,
        stage_name=stage_name,
        normalization_label=normalization_label,
        exclude_s10=exclude_s10,
        output_dir=pattern_dir
    )
    print(f"\nAnalisi dei pattern regionali completata. Risultati salvati in {pattern_dir}")
    
    # Step 7: Analizza le dinamiche delle fasi per regione
    print(f"\n===== ANALISI DELLE DINAMICHE DI FASE REGIONALI =====")
    phase_dir = os.path.join(output_base_dir, "phase_analysis")
    phase_results = analyze_phase_dynamics_by_region(
        group_regional_summaries,
        stage_name=stage_name,
        normalization_label=normalization_label,
        exclude_s10=exclude_s10,
        output_dir=phase_dir
    )
    print(f"\nAnalisi delle dinamiche di fase regionali completata. Risultati salvati in {phase_dir}")
    
    # Step 8: Stampa un riepilogo dei risultati
    print(f"\n===== RIEPILOGO DELL'ANALISI REGIONALE =====")
    for region in group_regional_summaries:
        print(f"\nRegione: {region}")
        if region in pattern_results["Region"].values:
            region_patterns = pattern_results[pattern_results["Region"] == region]
            print("  Analisi dei Pattern:")
            for _, row in region_patterns.iterrows():
                group = row["Group"]
                print(f"    {group}: {row['Pattern_Type']} (Picco: {row['Peak_Segment']}, Declino: {row['Post_Peak_Decline_Pct']:.1f}%)")
    
    print(f"\nTutti i risultati dell'analisi regionale sono stati salvati in: {output_base_dir}")
    
    # Restituisci tutti i risultati
    results = {
        "segment_data": segment_data,
        "regional_data": regional_data,
        "group_regional_summaries": group_regional_summaries,
        "pattern_results": pattern_results,
        "phase_results": phase_results
    }
    
    return results

# Esegui l'analisi se lo script viene eseguito direttamente
if __name__ == "__main__":
    # Parametri per l'analisi
    include_n2 = True           # True per N2+N3, False per solo N3
    normalize_nrem_only = True  # True per normalizzazione NREM, False per intera notte
    exclude_s10 = True          # True per escludere S10, False per includerlo
    
    # Esegui l'analisi multi-regionale
    results = main_multi_regional(
        include_n2=include_n2,
        normalize_nrem_only=normalize_nrem_only,
        exclude_s10=exclude_s10
    )
    
    print("\nAnalisi multi-regionale completata con successo!")


===== ANALYZING N2+N3 WITH NREM TIME MULTI-REGIONAL ANALYSIS =====


File in CTL - PD020:
['SO_power_none_c_E59.csv', 'SO_power_none_c_E183.csv', 'SO_power_none_f_E36.csv', 'PD020EEG_predicted_epochs.npy', 'PD020EEG_stages.npy', 'SO_power_none_f_E224.csv', 'SO_power_none_o_E116.csv']

File in CTL - PD010:
['SO_power_none_c_E59.csv', 'SO_power_none_c_E183.csv', 'PD010EEG_predicted_epochs.npy', 'SO_power_none_f_E36.csv', 'PD010EEG_stages.npy', 'SO_power_none_f_E224.csv', 'SO_power_none_o_E116.csv']

File in CTL - PD043:
['SO_power_none_c_E59.csv', 'SO_power_none_c_E183.csv', 'PD043EEG_predicted_epochs.npy', 'SO_power_none_f_E36.csv', 'PD043EEG_stages.npy', 'SO_power_none_f_E224.csv', 'SO_power_none_o_E116.csv']

File in CTL - PD033:
['SO_power_none_c_E59.csv', 'SO_power_none_c_E183.csv', 'SO_power_none_f_E36.csv', 'PD033EEG_stages.npy', 'PD033EEG_predicted_epochs.npy', 'SO_power_none_f_E224.csv', 'SO_power_none_o_E116.csv']

File in CTL - PD022:
['SO_power_none_c_E59.csv', 'SO_power_non

# TOPOGRAFIA    

In [40]:
import glob
import xml.etree.ElementTree as ET
import mne


def read_egi_electrode_coordinates(xml_file_path):
    """
    Legge le coordinate degli elettrodi dal file XML di EGI GSN.
    """
    try:
        # Parsing del file XML
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
        
        # Gestisce il namespace XML
        namespace = ''
        if '}' in root.tag:
            namespace = root.tag.split('}')[0] + '}'
        
        # Dizionario per memorizzare le coordinate
        coordinates = {}
        
        # Cerchiamo tutti i sensori
        sensor_path = f".//{namespace}sensor" if namespace else ".//sensor"
        number_path = f"{namespace}number" if namespace else "number"
        x_path = f"{namespace}x" if namespace else "x"
        y_path = f"{namespace}y" if namespace else "y"
        z_path = f"{namespace}z" if namespace else "z"
        
        for sensor in root.findall(sensor_path):
            number_element = sensor.find(number_path)
            
            if number_element is not None and number_element.text:
                number = int(number_element.text)
                
                # Creiamo il nome dell'elettrodo nel formato "E{numero}"
                electrode_name = f"E{number}"
                
                # Otteniamo le coordinate x, y, z direttamente
                x_elem = sensor.find(x_path)
                y_elem = sensor.find(y_path)
                z_elem = sensor.find(z_path)
                
                if x_elem is not None and y_elem is not None and z_elem is not None:
                    x = float(x_elem.text)
                    y = float(y_elem.text)
                    z = float(z_elem.text)
                    
                    # Aggiungiamo al dizionario
                    coordinates[electrode_name] = (x, y, z)
        
        print(f"Lette coordinate per {len(coordinates)} elettrodi")
        return coordinates
    
    except Exception as e:
        print(f"Errore nella lettura del file XML: {e}")
        return {}

def divide_to_regions():
    """
    Definisce i 150 elettrodi dello scalpo sull'area coperta dal sistema EEG 10-20
    Output:
    - regions: dizionario con nomi delle regioni cerebrali e relativi canali EEG
    - all_electrodes: lista di tutti i 150 canali EEG dello scalpo
    """
    # -----------------------------------------  Pre-frontal brain region  -----------------------------------------
    fp = np.sort(np.array([27, 33, 34, 38, 39, 47, 48, 26, 20, 19, 12, 11, 3, 2, 222]))
    # -------------------------------------------  Frontal brain region  -------------------------------------------
    f = np.sort(np.array([16, 22, 23, 24, 28, 29, 30, 35, 36, 40, 41, 42, 49, 50, 21, 15, 7, 14, 6, 207, 13, 5, 215,
                          4, 224, 223, 214, 206, 213, 205]))
    # -------------------------------------------  Central brain region  -------------------------------------------
    c = np.sort(np.array([9, 17, 43, 44, 45, 51, 52, 53, 57, 58, 59, 60, 64, 65, 66, 71, 72, 8, 257, 81, 186, 198,
                          197, 185, 132, 196, 184, 144, 204, 195, 183, 155, 194, 182, 164, 181, 173]))
    # -------------------------------------------  Temporal brain region  ------------------------------------------
    t = np.sort(np.array([55, 56, 62, 63, 69, 70, 74, 75, 84, 85, 96, 221, 212, 211, 203, 202, 193, 192, 180, 179,
                          171, 170]))
    # -------------------------------------------  Parietal brain region  ------------------------------------------
    p = np.sort(np.array([76, 77, 78, 79, 80, 86, 87, 88, 89, 97, 98, 99, 100, 110, 90, 101, 119, 172, 163, 154,
                          143, 131, 162, 153, 142, 130, 161, 152, 141, 129, 128]))
    # ------------------------------------------  Occipital brain region  ------------------------------------------
    o = np.sort(np.array([107, 108, 109, 116, 117, 118, 125, 126, 160, 151, 140, 150, 139, 127, 138]))

    # Definisci le regioni come dizionario
    regions = {
        'Prefrontal': ['E' + str(ch) for ch in fp],
        'Frontal': ['E' + str(ch) for ch in f],
        'Central': ['E' + str(ch) for ch in c],
        'Temporal': ['E' + str(ch) for ch in t],
        'Parietal': ['E' + str(ch) for ch in p],
        'Occipital': ['E' + str(ch) for ch in o]
    }

    # Crea una lista piatta di tutti gli elettrodi
    all_electrodes = []
    for region_electrodes in regions.values():
        all_electrodes.extend(region_electrodes)

    return regions, all_electrodes

def create_mne_info_from_coordinates(coordinates_3d, sfreq=128):
    """
    Crea un oggetto MNE Info dai dati delle coordinate degli elettrodi.
    """
    # Ottieni la lista degli elettrodi validi dalle regioni
    _, all_electrodes = divide_to_regions()
    valid_electrodes = set(all_electrodes)
    
    # Filtra i nomi dei canali per includere solo quelli validi
    ch_names = [name for name in sorted(coordinates_3d.keys()) if name in valid_electrodes]
    
    print(f"Usando {len(ch_names)} elettrodi validi su {len(coordinates_3d)} trovati nel file XML")
    
    # Prepara le coordinate per MNE
    ch_pos = {}
    for name in ch_names:
        if name in coordinates_3d:
            ch_pos[name] = coordinates_3d[name]
    
    # Crea un montaggio personalizzato
    montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
    
    # Crea un oggetto Info di MNE
    info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types='eeg')
    
    # Applica il montaggio all'info
    info.set_montage(montage)
    
    return info

def read_sleep_stages(file_path, stage_map):
    """
    Legge i dati degli stadi del sonno da un file .npy
    """
    try:
        # Carica i dati dal file .npy
        stages = np.load(file_path)
        
        # Converti in DataFrame
        stage_df = pd.DataFrame({
            "Time (s)": np.arange(0, len(stages) * 30, 30),  # Assumiamo epoche di 30 secondi
            "Sleep Stage": stages
        })
        
        # Converti i codici degli stadi in etichette
        if stage_map:
            stage_df["Sleep Stage Label"] = stage_df["Sleep Stage"].map(stage_map)
        
        return stage_df
    
    except Exception as e:
        print(f"Errore nella lettura del file degli stadi: {e}")
        return None

def read_so_power_data(file_path):
    """
    Legge i dati della potenza SO da un file CSV
    """
    try:
        # Carica i dati dal file CSV
        so_data = pd.read_csv(file_path)
        
        # Rinomina le colonne se necessario
        if "Time" in so_data.columns and "SO-Power" not in so_data.columns:
            so_data = so_data.rename(columns={"Time": "Time (s)"})
            
            # Cerca una colonna che potrebbe contenere la potenza SO
            power_columns = [col for col in so_data.columns if "power" in col.lower() or "so" in col.lower()]
            if power_columns:
                so_data = so_data.rename(columns={power_columns[0]: "SO-Power"})
        
        return so_data
    
    except Exception as e:
        print(f"Errore nella lettura del file della potenza SO: {e}")
        return None

def load_all_subjects_multi_channel(base_path, groups, sleep_stage_map, channel_list):
    """
    Funzione per caricare dati da più canali per ogni soggetto
    """
    all_data = {group: {} for group in groups}
    
    for group in groups:
        group_path = os.path.join(base_path, group)
        
        if not os.path.exists(group_path):
            print(f"Group path {group_path} does not exist")
            continue
        
        # List all subjects in this group
        subject_dirs = [d for d in os.listdir(group_path) 
                      if os.path.isdir(os.path.join(group_path, d))]
        
        for subject in subject_dirs:
            subject_path = os.path.join(group_path, subject)
            
            # Trova il file degli stadi in modo più flessibile
            stages_files = [f for f in os.listdir(subject_path) if f.endswith('_stages.npy')]
            
            if not stages_files:
                print(f"Sleep stages file missing for {group} - {subject}")
                continue
            
            stages_file = os.path.join(subject_path, stages_files[0])
            
            # Carica i dati degli stadi
            stages_data = read_sleep_stages(stages_file, sleep_stage_map)
            if stages_data is None:
                print(f"Error loading sleep stages for {group} - {subject}")
                continue
            
            # Inizializza il dizionario per questo soggetto
            all_data[group][subject] = {
                "sleep_stages": stages_data,
                "channels": {}
            }
            
            # Carica i dati per ciascun canale
            for channel in channel_list:
                # Cerca file con pattern più flessibile
                so_file_patterns = [
                    f"SO_power_none_f_{channel}.csv",
                    f"SO_power_none_c_{channel}.csv",
                    f"SO_power_none_o_{channel}.csv",
                    f"*SO_power*{channel}*.csv"  # Pattern più generale
                ]
                
                found_file = False
                for pattern in so_file_patterns:
                    matching_files = glob.glob(os.path.join(subject_path, pattern))
                    if matching_files:
                        so_file = matching_files[0]
                        so_data = read_so_power_data(so_file)
                        if so_data is not None:
                            all_data[group][subject]["channels"][channel] = so_data
                            print(f"Loaded data for {group} - {subject} - {channel} from {os.path.basename(so_file)}")
                            found_file = True
                            break
                
                if not found_file:
                    print(f"SO power file missing for {group} - {subject} - {channel}")
            
            # Se non ci sono dati di canale, rimuovi questo soggetto
            if not all_data[group][subject]["channels"]:
                del all_data[group][subject]
                print(f"No channel data available for {group} - {subject}, removed from dataset")
    
    return all_data

def normalize_time_multi_channel(subject_data, normalize_nrem_only=False, nrem_stages=[1, 2, 3]):
    """
    Funzione per normalizzare dati di più canali
    """
    for group in subject_data:
        for subject, data in subject_data[group].items():
            # Get sleep stages data
            stages_data = data["sleep_stages"].copy()
            
            # Normalize each channel
            for channel, so_data in data["channels"].items():
                so_data = so_data.copy()
                
                # Filter to times in common between SO power and sleep stages
                min_time = so_data["Time (s)"].min()
                max_time = so_data["Time (s)"].max()
                
                stages_filtered = stages_data[
                    (stages_data["Time (s)"] >= min_time) & 
                    (stages_data["Time (s)"] <= max_time)
                ].copy()
                
                # Get times for normalization (NREM only or entire night)
                if normalize_nrem_only:
                    # Filter to only NREM stages
                    normalization_epochs = stages_filtered[stages_filtered["Sleep Stage"].isin(nrem_stages)]
                    print(f"{group} - {subject} - {channel}: Using {len(normalization_epochs)} NREM epochs for time normalization")
                    
                    # Skip if no NREM data
                    if len(normalization_epochs) == 0:
                        print(f"No NREM data found for {group} - {subject} - {channel}, skipping normalization")
                        continue
                    
                    # Get NREM times for normalization
                    normalization_times = normalization_epochs["Time (s)"].values
                    # Sort the times
                    normalization_times = np.sort(normalization_times)
                    
                    # Create a mapping from time to sequence (in NREM)
                    sequence_map = {t: i for i, t in enumerate(normalization_times)}
                    
                    # Map SO power times to NREM sequence
                    so_data["NREM_Sequence"] = so_data["Time (s)"].map(lambda t: sequence_map.get(t, np.nan))
                    
                    # Drop rows with NaN sequence (not in NREM)
                    so_data = so_data.dropna(subset=["NREM_Sequence"])
                    
                    # Normalize the sequence to 0-100%
                    so_data["Normalized Time"] = (so_data["NREM_Sequence"] / (len(normalization_times) - 1)) * 100
                    
                    # Also normalize stages data for this channel
                    stages_filtered["NREM_Sequence"] = stages_filtered["Time (s)"].map(lambda t: sequence_map.get(t, np.nan))
                    stages_filtered = stages_filtered.dropna(subset=["NREM_Sequence"])
                    stages_filtered["Normalized Time"] = (stages_filtered["NREM_Sequence"] / (len(normalization_times) - 1)) * 100
                    
                    normalization_label = "NREM Time"
                else:
                    # Full night normalization
                    min_time = so_data["Time (s)"].min()
                    max_time = so_data["Time (s)"].max()
                    total_duration = max_time - min_time
                    
                    # Create normalized time (0-100%) for SO power
                    so_data["Normalized Time"] = ((so_data["Time (s)"] - min_time) / total_duration) * 100
                    
                    # Create normalized time for sleep stages
                    stages_filtered["Normalized Time"] = ((stages_filtered["Time (s)"] - min_time) / total_duration) * 100
                    
                    normalization_label = "Full Night"
                
                # Update the channel data
                subject_data[group][subject]["channels"][channel] = so_data
                
                # Store normalized stages data for this channel
                subject_data[group][subject][f"stages_{channel}"] = stages_filtered
    
    return subject_data, normalization_label

def calculate_deep_sleep_segment_power_multi_channel(subject_data, num_segments=10, include_n2=False):
    """
    Funzione per calcolare la potenza del segmento per più canali
    """
    # Initialize data structure for all channels
    deep_sleep_segment_data = {group: {} for group in subject_data}
    
    # Define which sleep stages to include as "Deep Sleep"
    if include_n2:
        deep_sleep_values = [2, 3]  # Include both N2 and N3
        stage_name = "N2+N3"
    else:
        deep_sleep_values = [3]     # Only N3
        stage_name = "N3"
    
    print(f"Analyzing SO power in {stage_name} across {num_segments} segments for all channels")
    
    for group in subject_data:
        for subject, data in subject_data[group].items():
            deep_sleep_segment_data[group][subject] = {}
            
            # Get sleep stages data
            for channel in data["channels"]:
                # Get SO power data for this channel
                so_data = data["channels"][channel].copy()
                
                # Get corresponding normalized stages data
                if f"stages_{channel}" in data:
                    stages_data = data[f"stages_{channel}"].copy()
                else:
                    print(f"No normalized sleep stages found for {group} - {subject} - {channel}, skipping")
                    continue
                
                # Create segment labels for both datasets
                segment_size = 100 / num_segments
                so_data["Segment"] = (so_data["Normalized Time"] // segment_size).astype(int)
                stages_data["Segment"] = (stages_data["Normalized Time"] // segment_size).astype(int)
                
                # Handle edge case for exactly 100%
                so_data.loc[so_data["Segment"] == num_segments, "Segment"] = num_segments - 1
                stages_data.loc[stages_data["Segment"] == num_segments, "Segment"] = num_segments - 1
                
                # Find Deep Sleep epochs based on clinical scoring
                deep_sleep_epochs = stages_data[stages_data["Sleep Stage"].isin(deep_sleep_values)]
                
                # Create a lookup dictionary for segment -> Deep Sleep status
                deep_sleep_time_dict = {}
                for _, row in deep_sleep_epochs.iterrows():
                    deep_sleep_time_dict[row["Time (s)"]] = True
                
                # Mark SO power points that occur during Deep Sleep
                so_data["Is_Deep_Sleep"] = so_data["Time (s)"].apply(lambda t: deep_sleep_time_dict.get(t, False))
                
                # Calculate average Deep Sleep power for each segment
                deep_sleep_data = so_data[so_data["Is_Deep_Sleep"] == True]
                
                if len(deep_sleep_data) > 0:  # Skip if no Deep Sleep data
                    segment_means = deep_sleep_data.groupby("Segment")["SO-Power"].mean()
                    segment_counts = deep_sleep_data.groupby("Segment")["SO-Power"].count()
                    
                    # Only keep segments with enough data points (e.g., at least 10)
                    valid_segments = segment_counts[segment_counts >= 10].index
                    segment_means = segment_means[segment_means.index.isin(valid_segments)]
                    
                    # Store the results for this channel
                    deep_sleep_segment_data[group][subject][channel] = segment_means
                else:
                    print(f"No {stage_name} data found for {group} - {subject} - {channel}")
    
    return deep_sleep_segment_data, stage_name

def average_segments_by_phase(segment_data, exclude_s10=True):
    """
    Calcola la media dei segmenti per le fasi early, mid e late
    """
    phase_data = {'early': {}, 'mid': {}, 'late': {}}
    
    # Per ciascun gruppo
    for group in segment_data:
        phase_data['early'][group] = {}
        phase_data['mid'][group] = {}
        phase_data['late'][group] = {}
        
        # Per ciascun soggetto nel gruppo
        for subject in segment_data[group]:
            # Per ciascun canale del soggetto
            for channel in segment_data[group][subject]:
                # Inizializzazione delle liste per i valori di ciascuna fase per questo canale
                if channel not in phase_data['early'][group]:
                    phase_data['early'][group][channel] = []
                if channel not in phase_data['mid'][group]:
                    phase_data['mid'][group][channel] = []
                if channel not in phase_data['late'][group]:
                    phase_data['late'][group][channel] = []
                
                # Ottieni i dati dei segmenti per questo canale
                segment_series = segment_data[group][subject][channel]
                
                # Converti in lista per facile manipolazione
                segments = list(segment_series.items())
                segments.sort()  # Ordina per indice del segmento
                
                if exclude_s10 and len(segments) > 9:
                    segments = segments[:9]  # Escludi S10
                
                num_segments = len(segments)
                early_end = num_segments // 3
                mid_end = 2 * (num_segments // 3)
                
                # Early phase (S1-S3)
                early_values = [val for _, val in segments[:early_end]]
                if early_values:
                    # Calcola la media per questo soggetto e aggiungi alla lista
                    phase_data['early'][group][channel].append(np.mean(early_values))
                
                # Mid phase (S4-S6)
                mid_values = [val for _, val in segments[early_end:mid_end]]
                if mid_values:
                    phase_data['mid'][group][channel].append(np.mean(mid_values))
                
                # Late phase (S7-S9)
                late_values = [val for _, val in segments[mid_end:]]
                if late_values:
                    phase_data['late'][group][channel].append(np.mean(late_values))
    
    # Calcola le medie per gruppo e canale
    for phase in phase_data:
        for group in phase_data[phase]:
            for channel in phase_data[phase][group]:
                values = phase_data[phase][group][channel]
                if values:
                    phase_data[phase][group][channel] = np.mean(values)
                else:
                    phase_data[phase][group][channel] = np.nan
    
    return phase_data

def create_mne_grid_topographic_maps(mne_info, phase_data, electrode_of_interest, regions_map,
                                   groups=None, output_dir="./topographic_grid"):
    """
    Create Topographic Maps Grid of Slow Oscillations Power Dynamics during NREM Phases
    
    Parameters:
    - mne_info: MNE Info object with electrode information
    - phase_data: Dictionary containing SO power data for different phases and groups
    - electrode_of_interest: Dictionary of selected electrodes and their regions
    - regions_map: Dictionary of brain regions and their electrodes
    - groups: List of groups to include (default: None, which uses all available groups)
    - output_dir: Directory to save the output figure
    
    Returns:
    - Boolean indicating successful figure creation
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # If no groups specified, use all available
    if groups is None:
        groups = []
        for phase in phase_data:
            groups.extend(list(phase_data[phase].keys()))
        groups = sorted(list(set(groups)))
    
    # Ensure groups are in the desired order: CTL, DNV, ADV, DYS
    desired_order = ["CTL", "DNV", "ADV", "DYS"]
    groups = [g for g in desired_order if g in groups]
    
    # Get list of electrodes of interest
    electrodes_of_interest = list(electrode_of_interest.keys())
    
    # Phases to visualize
    phases = ['early', 'mid', 'late']
    
    # Create figure with increased size for better readability
    fig = plt.figure(figsize=(24, 22))
    
    # Create a grid spec for more control
    gs = fig.add_gridspec(len(groups)+1, len(phases) + 1, 
                          height_ratios=[0.2] + [1]*len(groups),  # First row for phase labels 
                          width_ratios=[0.1, 1, 1, 1],  # First column for group labels
                          wspace=0.3, 
                          hspace=0.3)
    
    # Add phase labels on top
    phase_labels = {
        'early': 'Early\n(S1-S3)',
        'mid': 'Mid\n(S4-S6)',
        'late': 'Late\n(S7-S9)'
    }
    for j, phase in enumerate(phases):
        ax_phase = fig.add_subplot(gs[0, j+1])
        ax_phase.text(0.5, 0.5, phase_labels[phase], 
                      ha='center', va='center', 
                      fontsize=18, fontweight='bold')
        ax_phase.axis('off')
    
    # For each group and phase
    for i, group in enumerate(groups):
        # Add group label in the first column
        ax_group = fig.add_subplot(gs[i+1, 0])
        ax_group.text(0.5, 0.5, group, 
                      fontsize=18, 
                      fontweight='bold', 
                      ha='center', 
                      va='center',
                      rotation=90)
        ax_group.axis('off')
        
        # Topographic maps for each phase
        for j, phase in enumerate(phases):
            ax = fig.add_subplot(gs[i+1, j+1])
            
            # Prepare data for only electrodes of interest
            electrode_values = {}
            for channel in electrodes_of_interest:
                if group in phase_data[phase] and channel in phase_data[phase][group]:
                    electrode_values[channel] = phase_data[phase][group][channel]
            
            # Extract coordinates for only electrodes of interest
            ch_pos = {}
            for ch in electrode_values.keys():
                ch_pos[ch] = mne_info.get_montage().get_positions()['ch_pos'][ch]
            
            # Create subset of info with only electrodes of interest
            subset_info = mne.create_info(list(electrode_values.keys()), sfreq=128, ch_types='eeg')
            montage = mne.channels.make_dig_montage(ch_pos=ch_pos, coord_frame='head')
            subset_info.set_montage(montage)
            
            # Convert values to array
            data = np.array(list(electrode_values.values()))
            
            # Calculate global vmin and vmax
            all_values = []
            for phase_data_group in phase_data.values():
                for group_data in phase_data_group.values():
                    all_values.extend([val for ch, val in group_data.items() if ch in electrodes_of_interest])
            
            vmin, vmax = min(all_values), max(all_values)
            
            # Create topographic map
            im, _ = mne.viz.plot_topomap(data, 
                                          subset_info, 
                                          axes=ax, 
                                          show=False,
                                          cmap='viridis',  
                                          vlim=(vmin, vmax),
                                          extrapolate='head',
                                          outlines='head',  # Include head, nose, and ears
                                          sensors=True,    # Show electrode positions
                                          contours=6)
    
    # Add global colorbar
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    cbar = fig.colorbar(im, cax=cbar_ax)
    cbar.set_label('Slow Oscillations Power (dB)', fontsize=14)
    
    # Add main title 
    fig.suptitle("Slow Oscillations Power Dynamics during NREM Phases", 
               fontsize=24, 
               y=0.98, 
               fontweight='bold')
    
    # Save the figure
    plt.savefig(os.path.join(output_dir, "slow_oscillations_topographic_grid.png"), 
                dpi=300, bbox_inches="tight")
    plt.close(fig)
    
    print(f"Topographic grid map saved to {os.path.join(output_dir, 'slow_oscillations_topographic_grid.png')}")
    
    return True

def main():
    """
    Funzione principale che coordina il caricamento, elaborazione e visualizzazione dei dati
    """
    # Ottieni le regioni e tutti i canali
    regions_map, all_electrodes = divide_to_regions()
    
    # Definizione dei nostri elettrodi di interesse e le loro regioni
    electrode_of_interest = {
        'E36': 'Frontal',
        'E224': 'Frontal',
        'E59': 'Central',
        'E183': 'Central',
        'E116': 'Occipital'
    }
    
    # Percorso al file XML delle coordinate
    xml_file_path = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/clinical_scorings/coordinates.xml"
    
    # Percorso base dei clinical scoring
    base_path = "/Users/matilde_oropallo/PycharmProjects/pyDYNAM-O/data/clinical_scorings"
    
    # Definisci i gruppi
    groups = ["CTL", "DNV", "ADV", "DYS"]
    
    # Definisci la mappatura degli stadi del sonno
    sleep_stage_map = {
        0: "Wake",
        1: "N1",
        2: "N2",
        3: "N3", 
        5: "REM",
        6: "Artifacts"
    }
    
    # Parametri per l'analisi
    include_n2 = True           # True per N2+N3, False per solo N3
    normalize_nrem_only = True  # True per normalizzazione NREM, False per intera notte
    exclude_s10 = True          # True per escludere S10, False per includerlo
    num_segments = 10           # Numero di segmenti in cui dividere i dati
    
    # Setup output directory
    output_base_dir = "./topographic_results"
    stage_name = "N2+N3" if include_n2 else "N3"
    output_dir = os.path.join(output_base_dir, f"{stage_name.lower().replace('+', '_')}")
    grid_dir = os.path.join(output_dir, "grid")
    os.makedirs(grid_dir, exist_ok=True)
    
    # Elenco dei canali da analizzare
    channel_list = list(electrode_of_interest.keys())
    print(f"Canali da analizzare: {', '.join(channel_list)}")
    
    # Step 1: Carica tutti i dati dei soggetti per tutti i canali
    print("Caricamento dati dei soggetti e stadi clinici del sonno per tutti i canali...")
    subject_data = load_all_subjects_multi_channel(base_path, groups, sleep_stage_map, channel_list)
    
    # Stampa il riepilogo dei dati caricati
    for group in subject_data:
        print(f"Gruppo {group}: {len(subject_data[group])} soggetti caricati con dati completi")
        for subject in subject_data[group]:
            print(f"  - {subject}: {len(subject_data[group][subject]['channels'])} canali disponibili")
    
    # Step 2: Normalizza il tempo per tutti i canali
    print(f"\nNormalizzazione delle scale temporali utilizzando {'NREM time' if normalize_nrem_only else 'full night'} per tutti i canali...")
    nrem_stages = [1, 2, 3]  # N1, N2, N3
    normalized_data, normalization_label = normalize_time_multi_channel(subject_data, normalize_nrem_only=normalize_nrem_only, nrem_stages=nrem_stages)
    
    # Step 3: Calcola la potenza media del sonno profondo in ciascun segmento per ogni canale
    print(f"\nCalcolo delle medie di potenza {'N2+N3' if include_n2 else 'N3 only'} per {num_segments} segmenti in tutti i canali...")
    segment_data, stage_name = calculate_deep_sleep_segment_power_multi_channel(normalized_data, num_segments=num_segments, include_n2=include_n2)
    
    # Step 4: Calcola le medie per fasi (early, mid, late)
    print("Calcolo delle medie per fase (Early, Mid, Late)...")
    phase_data = average_segments_by_phase(segment_data, exclude_s10=exclude_s10)
    
    # Step 5: Leggi le coordinate degli elettrodi
    print("Lettura delle coordinate degli elettrodi...")
    coordinates_3d = read_egi_electrode_coordinates(xml_file_path)
    
    # Step 6: Crea un oggetto MNE Info con le coordinate degli elettrodi
    print("Creazione dell'oggetto MNE Info con le coordinate degli elettrodi...")
    mne_info = create_mne_info_from_coordinates(coordinates_3d)
    
    # Step 7: Crea la griglia di mappe topografiche con MNE
    print("Creazione della griglia di mappe topografiche con MNE...")
    create_mne_grid_topographic_maps(mne_info, phase_data, electrode_of_interest, regions_map,
                                    groups=groups, output_dir=grid_dir)
    
    print(f"Visualizzazione topografica completata. Risultati salvati in: {grid_dir}")

if __name__ == "__main__":
    main()

Canali da analizzare: E36, E224, E59, E183, E116
Caricamento dati dei soggetti e stadi clinici del sonno per tutti i canali...
Loaded data for CTL - PD020 - E36 from SO_power_none_f_E36.csv
Loaded data for CTL - PD020 - E224 from SO_power_none_f_E224.csv
Loaded data for CTL - PD020 - E59 from SO_power_none_c_E59.csv
Loaded data for CTL - PD020 - E183 from SO_power_none_c_E183.csv
Loaded data for CTL - PD020 - E116 from SO_power_none_o_E116.csv
Loaded data for CTL - PD010 - E36 from SO_power_none_f_E36.csv
Loaded data for CTL - PD010 - E224 from SO_power_none_f_E224.csv
Loaded data for CTL - PD010 - E59 from SO_power_none_c_E59.csv
Loaded data for CTL - PD010 - E183 from SO_power_none_c_E183.csv
Loaded data for CTL - PD010 - E116 from SO_power_none_o_E116.csv
Loaded data for CTL - PD043 - E36 from SO_power_none_f_E36.csv
Loaded data for CTL - PD043 - E224 from SO_power_none_f_E224.csv
Loaded data for CTL - PD043 - E59 from SO_power_none_c_E59.csv
Loaded data for CTL - PD043 - E183 from 